In [1]:
import MarsGT 
from MarsGT.conv import *
from MarsGT.egrn import *
from MarsGT.marsgt_model import *
from MarsGT.utils import *
import anndata as ad
from collections import Counter
import copy
import dill
from functools import partial
import json
import math
import matplotlib.cm as cm
import matplotlib.pyplot as plt
import multiprocessing as mp
import numpy as np
import os
import pandas as pd
from operator import itemgetter
import random
import scipy.sparse as sp
from scipy.io import mmread
from scipy.sparse import hstack, vstack, coo_matrix
import seaborn as sb
from sklearn import metrics
from sklearn.cluster import KMeans
from sklearn.decomposition import IncrementalPCA
from sklearn.decomposition import SparsePCA
from sklearn.metrics import accuracy_score
from sklearn.metrics.cluster import normalized_mutual_info_score
import time
import torch
import torch.cuda as cuda
from torch import nn
from torch.autograd import Variable
import torch.distributions as D
import torch.nn.functional as F
import torch_geometric.data as Data
from torch_geometric.nn import GCNConv, GATConv
from torch_geometric.nn.conv import MessagePassing
from torch_geometric.nn.inits import glorot, uniform
from torch_geometric.utils import softmax as Softmax
from torchmetrics.functional import pairwise_cosine_similarity
import warnings
from warnings import filterwarnings
import xlwt
import argparse
from tqdm import tqdm
import scanpy as sc

In [2]:
parser = argparse.ArgumentParser(description='Training GNN on gene cell graph')
parser.add_argument('--fi', type=int, default=0) # This parameter is used for the benchmark to specify the starting sequence number of the created files
parser.add_argument('--labsm', type=float, default=0.1) # The rate of LabelSmoothing
parser.add_argument('--wd', type=float, default=0.1) # The 'weight_decay' parameter is used to specify the strength of L2 regularization
parser.add_argument('--lr', type=float, default=0.0005) # learning rate
parser.add_argument('--n_hid', type=int, default=104) # The number of layers should be a multiple of 'n_head' in order to make any modifications
parser.add_argument('--nheads', type=int, default=8) # The 'heads' parameter represents the number of attention heads in the attention mechanism
parser.add_argument('--nlayers', type=int, default=3) # The number of layers in network
parser.add_argument('--cell_size', type=int, default=30) # The number of cells per subgraph (batch)
parser.add_argument('--neighbor', type=int, default=20) # The number of neighboring nodes to be selected for each cell in the subgraph
parser.add_argument('--egrn', type=bool, default=False) # Whether to output the Enhancer-Gene regulatory network
parser.add_argument('--output_file', type=str, default='/home/jsl/YBR/Experiment/test_marsgt') # Please replace the actual path with the path you want. 
args = parser.parse_args([])

output_file = args.output_file
fi=args.fi
labsm = args.labsm
lr = args.lr
wd = args.wd
n_hid = args.n_hid
nheads = args.nheads
nlayers = args.nlayers
cell_size = args.cell_size
neighbor = args.neighbor
egrn = args.egrn

In [1]:
os.chdir("./ComicGTN/data/BMMC-bench-1")
# RNA_cell_label = pd.read_csv('Cell_types.tsv', sep='\t', header=None)
gene_peak = ad.read_mtx('Gene_Peak.mtx')
gene_cell = ad.read_mtx('Gene_Cell.mtx')
peak_cell = ad.read_mtx('Peak_Cell.mtx')
gene_names = pd.read_csv('Gene_names.tsv', sep='\t', header=None)
cell_names = pd.read_csv('Cell_names.tsv', sep='\t', header=None)
peak_names = pd.read_csv('Peak_names.tsv', sep='\t', header=None)

# true_label = np.array(RNA_cell_label[0])
peak_cell.obs_names = peak_names[0]
peak_cell.var_names = cell_names[0]
gene_cell.obs_names = gene_names[0]
gene_cell.var_names = cell_names[0]
gene_peak.obs_names = gene_names[0]
gene_peak.var_names = peak_names[0]

RNA_matrix = gene_cell.X
ATAC_matrix = peak_cell.X
RP_matrix = gene_peak.X
Gene_Peak = gene_peak.X

cell_num = RNA_matrix.shape[1]
gene_num = RNA_matrix.shape[0]
peak_num = ATAC_matrix.shape[0]

In [7]:
if __name__ == "__main__":
    device = torch.device("cuda" if cuda.is_available() else "cpu")
    print('You will use : ',device)
    # clustering result by scanpy
    initial_pre = initial_clustering(RNA_matrix) 
    # number of every cluster
    cluster_ini_num = len(set(initial_pre)) 
    ini_p1 = [int(i) for i in initial_pre] 
    # partite the data into batches
    indices, Node_Ids, dic = batch_select_whole (RNA_matrix, ATAC_matrix, neighbor = [neighbor], cell_size=cell_size)
    n_batch = len(indices)
    
    # Reduce the dimensionality of features for cell, gene, and peak data.
    node_model = NodeDimensionReduction(RNA_matrix, ATAC_matrix, indices, ini_p1, n_hid=n_hid, n_heads=nheads, 
                                        n_layers=nlayers,labsm=labsm, lr=lr, wd=wd, device=device, num_types=3, num_relations=2, epochs=1)
    gnn,cell_emb,gene_emb,peak_emb,h = node_model.train_model(n_batch=n_batch)

    # Instantiate the MarsGT_model
    MarsGT_model = MarsGT(gnn=gnn, h=h, labsm=labsm, n_hid=n_hid, n_batch=n_batch, device=device,lr=lr,wd=wd, num_epochs=1)
    # Train the model
    MarsGT_gnn = MarsGT_model.train_model(indices=indices,RNA_matrix=RNA_matrix, ATAC_matrix=ATAC_matrix, Gene_Peak=Gene_Peak, ini_p1=ini_p1)
    # The result of MarsGT
    MarsGT_result = MarsGT_pred(RNA_matrix, ATAC_matrix, RP_matrix, egrn=egrn, MarsGT_gnn=MarsGT_gnn, indices=indices, 
                        nodes_id=Node_Ids, cell_size=cell_size, device=device, gene_names=gene_names, peak_names=peak_names)
    
    MarsGT_result['pred_label'].to_csv("./ComicGTN/data/BMMC-bench-1/MarsGT_pred.tsv", sep="\t", header=False, index=False)

You will use :  cpu
	When the number of cells is less than or equal to 500, it is recommended to set the resolution value to 0.2.
	When the number of cells is within the range of 500 to 5000, the resolution value should be set to 0.5.
	When the number of cells is greater than 5000, the resolution value should be set to 0.8.
         Falling back to preprocessing with `sc.pp.pca` and default params.


/home/jsl/anaconda3/envs/marsgt/lib/python3.8/site-packages/numba/np/ufunc/parallel.py:371: NumbaWarning: The TBB threading layer requires TBB version 2021 update 6 or later i.e., TBB_INTERFACE_VERSION >= 12060. Found TBB_INTERFACE_VERSION = 12050. The TBB threading layer is disabled.
  warnings.warn(problem)


We are currently in the process of partitioning the data into batches. Kindly wait for a moment, please.


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 512/512 [32:21<00:00,  3.79s/it]


The training process for the NodeDimensionReduction model has started. Please wait.


  0%|                                                                                                                                   | 0/1 [00:00<?, ?it/s]/home/jsl/anaconda3/envs/marsgt/lib/python3.8/site-packages/MarsGT/marsgt_model.py:151: FutureWarning: Using a non-tuple sequence for multidimensional indexing is deprecated; use `arr[tuple(seq)]` instead of `arr[seq]`. In the future this will be interpreted as an array index, `arr[np.array(seq)]`, which will result either in an error or a different result.
  l = torch.LongTensor(np.array(self.ini_p1)[[cell_index]]).to(self.device)
/home/jsl/anaconda3/envs/marsgt/lib/python3.8/site-packages/torch/nn/functional.py:2904: UserWarning: reduction: 'mean' divides the total loss by both the batch size and the support size.'batchmean' divides only by the batch size, and aligns with the KL div math definition.'mean' will be changed to behave the same as 'batchmean' in the next major release.
  warnings.warn(
100%|██████████████████████████

The training for the NodeDimensionReduction model has been completed.
The training process for the MarsGT model has started. Please wait.


  0%|                                                                                                                                 | 0/512 [00:00<?, ?it/s]/home/jsl/anaconda3/envs/marsgt/lib/python3.8/site-packages/MarsGT/marsgt_model.py:251: FutureWarning: Using a non-tuple sequence for multidimensional indexing is deprecated; use `arr[tuple(seq)]` instead of `arr[seq]`. In the future this will be interpreted as an array index, `arr[np.array(seq)]`, which will result either in an error or a different result.
  l = torch.LongTensor(np.array(ini_p1)[[cell_index]]).to(self.device)
100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 512/512 [1:41:19<00:00, 11.87s/it]


The training for the MarsGT model has been completed.


  0%|                                                                                                                                 | 0/512 [00:00<?, ?it/s]/home/jsl/anaconda3/envs/marsgt/lib/python3.8/site-packages/MarsGT/marsgt_model.py:357: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at  ../torch/csrc/utils/tensor_new.cpp:201.)
  gene_cell_edge_index = torch.LongTensor(
100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 512/512 [22:41<00:00,  2.66s/it]
